In [154]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from sklearn.preprocessing import StandardScaler

In [155]:
train_df = pd.read_csv("./train.csv")
test_df = pd.read_csv("./test.csv")

In [156]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    object 
 3   Sex          418 non-null    object 
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    object 
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     object 
 10  Embarked     418 non-null    object 
dtypes: float64(2), int64(4), object(5)
memory usage: 36.1+ KB


In [157]:
train_df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [158]:
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [159]:
train_df.tail()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.00,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.00,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.45,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.00,C148,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,370376,7.75,NaN,Q


In [160]:
train_df.drop(columns=["Name", "Cabin", "Ticket"], inplace=True)
test_df.drop(columns=["Name", "Cabin", "Ticket"], inplace=True)
train_df["Sex"] = train_df["Sex"].map({"female": 0, "male": 1})
test_df["Sex"] = test_df["Sex"].map({"female": 0, "male": 1})
train_df["FamilySize"] = train_df["SibSp"] + train_df["Parch"]
test_df["FamilySize"] = test_df["SibSp"] + test_df["Parch"]

In [161]:
train_df.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize
0,1,0,3,1,22.0,1,0,7.2500,S,1
1,2,1,1,0,38.0,1,0,71.2833,C,1
2,3,1,3,0,26.0,0,0,7.9250,S,0
3,4,1,1,0,35.0,1,0,53.1000,S,1
4,5,0,3,1,35.0,0,0,8.0500,S,0


In [162]:
train_df = pd.get_dummies(data=train_df, columns=["Embarked"], drop_first=True)
test_df = pd.get_dummies(data=test_df,columns=["Embarked"],drop_first=True)
train_df.head()

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,FamilySize,Embarked_Q,Embarked_S
0,1,0,3,1,22.0,1,0,7.2500,1,False,True
1,2,1,1,0,38.0,1,0,71.2833,1,False,False
2,3,1,3,0,26.0,0,0,7.9250,0,False,True
3,4,1,1,0,35.0,1,0,53.1000,1,False,True
4,5,0,3,1,35.0,0,0,8.0500,0,False,True


In [ ]:
train_df["Age"].fillna(train_df["Age"].median(), inplace=True)
test_df["Age"].fillna(test_df["Age"].median(), inplace=True)
train_df.fillna(0, inplace=True)
test_df.fillna(0, inplace=True)

test_df["Age"] = test_df["Age"].astype(str).astype(float)
train_df = train_df.astype(float)

C:\Users\Haji Suleman\AppData\Local\Temp\ipykernel_10592\308710003.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df["Age"].fillna(train_df["Age"].median(), inplace=True)
C:\Users\Haji Suleman\AppData\Local\Temp\ipykernel_10592\308710003.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always b

In [164]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Sex          418 non-null    int64  
 3   Age          418 non-null    float64
 4   SibSp        418 non-null    int64  
 5   Parch        418 non-null    int64  
 6   Fare         418 non-null    float64
 7   FamilySize   418 non-null    int64  
 8   Embarked_Q   418 non-null    bool   
 9   Embarked_S   418 non-null    bool   
dtypes: bool(2), float64(2), int64(6)
memory usage: 27.1 KB


In [165]:
X = train_df.drop(columns=["Survived"])
y = train_df["Survived"]


scaler = StandardScaler()
X = X.fillna(0)
X_np = X.values
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X, y.to_numpy(), test_size=0.2, random_state=42
)

X_train_np = scaler.fit_transform(X_train_np)
X_test_np = scaler.transform(X_test_np)

X_train = torch.tensor(X_train_np, dtype=torch.float32)
X_test = torch.tensor(X_test_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.float32)

In [166]:
# 1. Force the whole column to strings (fixes the "Invalid object type" error)


In [167]:
class titanic_model(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(X_train.shape[1], 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, X):
        return self.model(X)

In [168]:
torch.manual_seed(42)
model = titanic_model()

## %% Loss function and optimizer

loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(params=model.parameters(),lr=0.001)

In [169]:
## Training Loop

epochs = 250
for epoch in range(epochs):
    model.train()
    X_preds = model(X_train)
    loss = loss_fn(X_preds, y_train.unsqueeze(dim=1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.eval()
    with torch.inference_mode():
        test_preds = model(X_test)
        pred_labels = (test_preds.squeeze() >= 0.5).float()
        acc = (pred_labels == y_test).float().mean()
        test_loss = loss_fn(test_preds, y_test.unsqueeze(dim=1))
        if epoch % 10 == 0:
            print(
                f"Epoch: {epoch}, Loss: {loss:.4f}, Test Loss: {test_loss:.4f}, Accuracy: {acc:.4f}"
            )

Epoch: 0, Loss: 0.6996, Test Loss: 0.6924, Accuracy: 0.5251
Epoch: 10, Loss: 0.6839, Test Loss: 0.6777, Accuracy: 0.5866
Epoch: 20, Loss: 0.6696, Test Loss: 0.6641, Accuracy: 0.6369
Epoch: 30, Loss: 0.6561, Test Loss: 0.6513, Accuracy: 0.6983
Epoch: 40, Loss: 0.6432, Test Loss: 0.6389, Accuracy: 0.7263
Epoch: 50, Loss: 0.6305, Test Loss: 0.6264, Accuracy: 0.7486
Epoch: 60, Loss: 0.6180, Test Loss: 0.6138, Accuracy: 0.7374
Epoch: 70, Loss: 0.6052, Test Loss: 0.6007, Accuracy: 0.7318
Epoch: 80, Loss: 0.5923, Test Loss: 0.5872, Accuracy: 0.7486
Epoch: 90, Loss: 0.5790, Test Loss: 0.5732, Accuracy: 0.7430
Epoch: 100, Loss: 0.5657, Test Loss: 0.5590, Accuracy: 0.7430
Epoch: 110, Loss: 0.5525, Test Loss: 0.5450, Accuracy: 0.7598
Epoch: 120, Loss: 0.5396, Test Loss: 0.5311, Accuracy: 0.7709
Epoch: 130, Loss: 0.5271, Test Loss: 0.5176, Accuracy: 0.7709
Epoch: 140, Loss: 0.5152, Test Loss: 0.5048, Accuracy: 0.8045
Epoch: 150, Loss: 0.5039, Test Loss: 0.4928, Accuracy: 0.7989
Epoch: 160, Loss: 0

In [170]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Sex          418 non-null    int64  
 3   Age          418 non-null    float64
 4   SibSp        418 non-null    int64  
 5   Parch        418 non-null    int64  
 6   Fare         418 non-null    float64
 7   FamilySize   418 non-null    int64  
 8   Embarked_Q   418 non-null    bool   
 9   Embarked_S   418 non-null    bool   
dtypes: bool(2), float64(2), int64(6)
memory usage: 27.1 KB


In [171]:
test_df.fillna(0, inplace=True)
test_df = test_df.astype(float)
passengers_id = test_df["PassengerId"].values
test_scaled = scaler.transform(test_df)
test_df = torch.from_numpy(test_scaled).type(torch.float32)

In [172]:
with torch.inference_mode():
    probs = model(test_df).squeeze()
    predictions = (probs >= 0.5).int().numpy()
submission = pd.DataFrame(
    {"PassengerId": passengers_id.astype(int), "Survived": predictions}
)
submission.to_csv("submission.csv", index=False)